## Line overlay inspector\n
\n
Interactive notebook to inspect per-frame polyline labels overlaid on images.\n
\n
- Images: `...\\images\\img0048505.png`\n
- Labels: `...\\labels\\<label_type>\\0048505.csv` (headerless CSV, each row is `x,y`)\n
\n
Set `DATASET_DIR` below to your dataset folder that contains `images/` and `labels/`.

In [ ]:
# If needed (run once):
# !pip install numpy pillow matplotlib ipywidgets

from pathlib import Path

import ipywidgets as W
from IPython.display import display, clear_output

from line_overlay_utils import DatasetPaths, list_frames_in_images_dir, list_label_types, load_polyline_csv, render_overlay_matplotlib

DATASET_DIR = r"C:\\Users\\wanglab\\Desktop\\Mel\\Mel_MRN_opto_cohort_videos\\TRAINING_DATA\\MRN_opto5_F_8_2_25_L_20260116_opto_xp_2s_dur_10x_stimulations0_nob"

paths = DatasetPaths(dataset_dir=Path(DATASET_DIR))
frames = list_frames_in_images_dir(paths.images_dir, image_prefix=paths.image_prefix, image_ext=paths.image_ext)
label_types = list_label_types(paths.labels_dir)

print(f"Images dir: {paths.images_dir}")
print(f"Labels dir: {paths.labels_dir}")
print(f"Found {len(frames)} frame(s) in images/")
print(f"Found label types: {label_types}")

In [ ]:
dataset_text = W.Text(
    value=DATASET_DIR,
    description="dataset",
    layout=W.Layout(width="900px"),
)
reload_btn = W.Button(description="reload", button_style="")

label_dropdown = W.Dropdown(
    options=label_types if label_types else ["0"],
    value=(label_types[0] if label_types else "0"),
    description="label",
    layout=W.Layout(width="250px"),
)

min_frame = min(frames) if frames else 0
max_frame = max(frames) if frames else 0

frame_slider = W.IntSlider(
    value=min_frame,
    min=min_frame,
    max=max_frame,
    step=1,
    description="frame",
    continuous_update=False,
    readout=True,
    layout=W.Layout(width="650px"),
)
frame_text = W.IntText(value=min_frame, description="frame", layout=W.Layout(width="250px"))
W.jslink((frame_slider, "value"), (frame_text, "value"))

show_points = W.Checkbox(value=False, description="show points")

status = W.HTML(value="")
out = W.Output()


def _load_state():
    p = DatasetPaths(dataset_dir=Path(dataset_text.value))
    fr = list_frames_in_images_dir(p.images_dir, image_prefix=p.image_prefix, image_ext=p.image_ext)
    lt = list_label_types(p.labels_dir)
    return p, fr, lt


def _update_ui_from_reload(*_):
    global paths, frames

    paths, frames, lts = _load_state()
    if not frames:
        frame_slider.min = 0
        frame_slider.max = 0
        frame_slider.value = 0
        frame_text.value = 0
    else:
        frame_slider.min = min(frames)
        frame_slider.max = max(frames)
        if frame_slider.value < frame_slider.min or frame_slider.value > frame_slider.max:
            frame_slider.value = frame_slider.min

    if lts:
        label_dropdown.options = lts
        if label_dropdown.value not in lts:
            label_dropdown.value = lts[0]
    else:
        label_dropdown.options = ["0"]
        label_dropdown.value = "0"

    status.value = (
        f"<pre>Images: {paths.images_dir}\nLabels: {paths.labels_dir}\nFrames: {len(frames)}\nLabel types: {list(label_dropdown.options)}</pre>"
    )
    _render()


def _render(*_):
    with out:
        clear_output(wait=True)

        frame = int(frame_slider.value)
        label_type = str(label_dropdown.value)

        img_path = paths.image_path(frame)
        csv_path = paths.label_csv_path(label_type, frame)

        if not img_path.exists():
            print(f"Missing image: {img_path}")
            return

        pts = load_polyline_csv(csv_path)
        if pts.shape[0] == 0:
            print(f"No label found for label={label_type}, frame={frame} (expected: {csv_path})")

        render_overlay_matplotlib(
            img_path,
            pts,
            alpha=0.5,
            show_points=bool(show_points.value),
        )


reload_btn.on_click(_update_ui_from_reload)
frame_slider.observe(_render, names="value")
label_dropdown.observe(_render, names="value")
show_points.observe(_render, names="value")

controls = W.VBox([
    W.HBox([dataset_text, reload_btn]),
    status,
    W.HBox([label_dropdown, show_points]),
    W.HBox([frame_slider, frame_text]),
])

display(controls, out)
_update_ui_from_reload()